In [1]:
# Imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

In [ ]:
Raw_Path = "FREDMD-Current.csv"
series = ["RPI", "UNRATE", "CPIAUCSL", "GS5", "DPCERA3M086SBEA"]

raw_csv = pd.read_csv(Raw_Path)
transform_codes = raw_csv.iloc[0]
data = raw_csv.iloc[1:].reset_index(drop=True)
data["sasdate"] = pd.to_datetime(data["sasdate"])
data = data.set_index("sasdate")
data = data.apply(pd.to_numeric, errors="coerce")
df = data[series].copy()

In [3]:
def transform(series, code):
    if code == 1:
        return series
    elif code == 2:
        return series.diff()
    elif code == 3:
        return series .diff().diff()
    elif code == 4:
        return np.log(series)
    elif code == 5:
        return np.log(series).diff()
    elif code == 6:
        return np.log(series).diff().diff()
    elif code == 7:
        return (series / series.shift(1) - 1).diff()
    else:
        return series

df = df.interpolate(method = "linear")

series_tracker = {}
for col in series:
    s = transform(df[col], transform_codes[col]).dropna()
    series_tracker[col] = s

In [4]:
def make_windows_volnorm(arr, p, vol_window=60):
    X, Y, vols = [], [], []
    for i in range(p, len(arr)):
        start = max(0, i - vol_window)
        vol = arr[start:i].std() + 1e-5     
        X.append(arr[i-p:i] / vol)         
        Y.append(arr[i] / vol)               
        vols.append(vol)                     
    X = np.array(X, dtype=np.float32)[..., None]   
    Y = np.array(Y, dtype=np.float32)[:, None]     
    vols = np.array(vols, dtype=np.float32)[:, None]
    return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(vols)

def make_windows_plain(arr, p):
    X, Y = [], []
    for i in range(p, len(arr)):
        X.append(arr[i-p:i])
        Y.append(arr[i])
    X = np.array(X, dtype=np.float32)[..., None]
    Y = np.array(Y, dtype=np.float32)[:, None]
    return torch.from_numpy(X), torch.from_numpy(Y)

def time_split_vn(X, Y, vols, train=0.70, val=0.10):
    n = len(X)
    tr = int(n * train); va = int(n * (train + val))
    return (X[:tr], Y[:tr], vols[:tr],
            X[tr:va], Y[tr:va], vols[tr:va],
            X[va:], Y[va:], vols[va:])

In [ ]:
# Loss, training (early stopping), evaluation with vol reversal
loss_function = nn.MSELoss()

def ar1(X_tr, Y_tr, X_te, Y_te):
    x_tr = X_tr[:, -1, 0].numpy(); y_tr = Y_tr[:, 0].numpy()
    A = np.vstack([x_tr, np.ones(len(x_tr))]).T
    a, b = np.linalg.lstsq(A, y_tr, rcond=None)[0]
    x_te = X_te[:, -1, 0].numpy()
    pred = a * x_te + b
    return float(np.mean((pred - Y_te[:, 0].numpy()) ** 2))

def train_model(model, X_tr, Y_tr, X_val, Y_val,
                num_epochs=500, learning_rate=0.01, patience=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_val = float("inf"); best_state = None; improve = 0
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_function(model(X_tr), Y_tr)   # trained on vol-normalized scale
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_function(model(X_val), Y_val)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            improve = 0
        else:
            improve += 1
        if improve >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def evaluate_vn(model, X, Y, vols):
    model.eval()
    with torch.no_grad():
        pred = model(X) * vols
        actual = Y * vols
        return loss_function(pred, actual).item()

In [ ]:
class LSTMAttention(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):                          
        hidden_states, (h_n, c_n) = self.lstm(x)   
        q = h_n[-1].unsqueeze(1)                   
        scores = torch.matmul(q, hidden_states.transpose(1, 2)) / (self.hidden_size ** 0.5)
        alpha = torch.softmax(scores, dim=-1)      
        context = torch.matmul(alpha, hidden_states).squeeze(1)  
        prediction = self.fc(context)               
        return prediction, alpha
    def predict(self, x):
        return self.forward(x)[0]

In [ ]:
# Run LSTM+Attention
lags = 12
seeds = [0, 1, 2, 3, 4]
vn_results_attn = {}

class AttnWrapper(nn.Module):
    def __init__(self, core): 
        super().__init__(); self.core = core
    def forward(self, x): 
        return self.core(x)[0]

for name in series:
    arr = series_tracker[name].values.astype(np.float32)
    X, Y, vols = make_windows_volnorm(arr, lags)
    (X_tr, Y_tr, v_tr, X_val, Y_val, v_val, X_te, Y_te, v_te) = time_split_vn(X, Y, vols)

    Xp, Yp = make_windows_plain(arr, lags)
    n = len(Xp); trp = int(n*0.70); vap = int(n*0.80)
    base_mse = ar1(Xp[:trp], Yp[:trp], Xp[vap:], Yp[vap:])

    errs = []
    for seed in seeds:
        set_seed(seed)
        model = AttnWrapper(LSTMAttention(input_size=1, hidden_size=50))
        model, _ = train_model(model, X_tr, Y_tr, X_val, Y_val)
        errs.append(evaluate_vn(model, X_te, Y_te, v_te))
    errs = np.array(errs)
    vn_results_attn[name] = {"baseline": base_mse, "mean": errs.mean(), "std": errs.std()}
    print(f"{name:18s} AR(1)={base_mse:.6f}  Attn(volnorm)={errs.mean():.6f} +/- {errs.std():.6f}")

table = pd.DataFrame(vn_results_attn).T
print("\n", table)

RPI                AR(1)=0.000471  Attn(volnorm)=0.000568 +/- 0.000011
UNRATE             AR(1)=0.781290  Attn(volnorm)=0.866979 +/- 0.006136
CPIAUCSL           AR(1)=0.000007  Attn(volnorm)=0.000006 +/- 0.000000
GS5                AR(1)=0.034780  Attn(volnorm)=0.036421 +/- 0.001833
DPCERA3M086SBEA    AR(1)=0.000209  Attn(volnorm)=0.000194 +/- 0.000002

                  baseline      mean           std
RPI              0.000471  0.000568  1.126263e-05
UNRATE           0.781290  0.866979  6.136243e-03
CPIAUCSL         0.000007  0.000006  1.484600e-07
GS5              0.034780  0.036421  1.833473e-03
DPCERA3M086SBEA  0.000209  0.000194  1.723653e-06


In [8]:
# Inspect attention weights (assignment requirement)
# Train one attention model and average its attention per lag position on the test set.
name = "GS5"   # change to inspect a different series
arr = series_tracker[name].values.astype(np.float32)
X, Y, vols = make_windows_volnorm(arr, lags)
(X_tr, Y_tr, v_tr, X_val, Y_val, v_val, X_te, Y_te, v_te) = time_split_vn(X, Y, vols)

set_seed(0)
core = LSTMAttention(input_size=1, hidden_size=50)   # the raw attention model (returns pred, alpha)
wrapped = AttnWrapper(core)                            # wrapper returns pred only, for training
wrapped, _ = train_model(wrapped, X_tr, Y_tr, X_val, Y_val)

core.eval()
with torch.no_grad():
    _, alpha = core(X_te)                              # alpha: (batch, 1, p)
avg_attention = alpha.squeeze(1).mean(dim=0).numpy()   # mean attention per lag position

print(f"Average attention per lag position ({name}):")
for lag_pos, w in enumerate(avg_attention, start=1):
    print(f"  lag {lag_pos:2d} (i.e. {lags - lag_pos + 1} months ago): {w:.3f}")

Average attention per lag position (GS5):
  lag  1 (i.e. 12 months ago): 0.020
  lag  2 (i.e. 11 months ago): 0.025
  lag  3 (i.e. 10 months ago): 0.031
  lag  4 (i.e. 9 months ago): 0.040
  lag  5 (i.e. 8 months ago): 0.050
  lag  6 (i.e. 7 months ago): 0.067
  lag  7 (i.e. 6 months ago): 0.079
  lag  8 (i.e. 5 months ago): 0.101
  lag  9 (i.e. 4 months ago): 0.112
  lag 10 (i.e. 3 months ago): 0.141
  lag 11 (i.e. 2 months ago): 0.141
  lag 12 (i.e. 1 months ago): 0.193
